In [ ]:
# --- repo bootstrap: make `panelclv` importable and relative data paths resolve ---
import os, sys
from pathlib import Path
_root = Path.cwd().resolve()
while _root != _root.parent and not (_root / 'pyproject.toml').exists():
    _root = _root.parent
os.chdir(_root)                                 # so "Datasets/..." resolve from repo root
_src = _root / 'src'
if _src.exists() and str(_src) not in sys.path:  # fallback if panelclv isn't pip-installed
    sys.path.insert(0, str(_src))
# ---------------------------------------------------------------------------------

%reload_ext autoreload
%autoreload 2

**Environment**

Run the notebook with the project venv (PyTorch on ROCm):

```
cd ~/Desktop/Thesis
source venvs/thesis_rocm/bin/activate
```

The project is now an installable package -- from the repo root run
`pip install -e .` once, after which `from panelclv.models import ...` works with no
manual path setup.

In [ ]:
# ---------------------------------------------------------------------------
# Third-party
# ---------------------------------------------------------------------------
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch

# ---------------------------------------------------------------------------
# This project (installed via `pip install -e .`, so no manual path setup).
# Models/__init__.py is the public API.
# ---------------------------------------------------------------------------
from panelclv.data_preparation import dynamic_panel_dataset
from panelclv.models import (
    MultinomialTransformerModel,  # softmax-over-counts Transformer (classifier head)
    compute_class_weights,  # class weights for weighted_ce / focal
    InferenceMultinomialTransformerModel,  # same backbone, sampling head
    mc_forecast_transformer,  # Transformer holdout rollout (growing-window, stateless)
    mc_compute_metrics,  # RMSE / bias% / aggregate-MAPE
)
from panelclv.training import (
    fit_model,  # shared training loop (CE / weighted_ce / focal / emd)
)
from panelclv.tuning import (
    run_optuna_study,  # hyper-parameter + covariate search
)
from panelclv.evaluation import (
    weekly_aggregate_predictions,
    plot_weekly_aggregated,
    metrics_table,
)
from panelclv.experiments import (
    make_data_builder,  # builds the Optuna data_builder closure
    build_inference_from_trial,  # rebuilds the winning model + loads its checkpoint
    refit_best_trial,  # warm-start retrain the winner on the full calibration window
)

# Data Loading


In [ ]:
from panelclv.configs.panel_config import PanelConfig

csv_path = "Datasets/Dataset_clean/electronics_customer_week_panel.csv"

# One config object replaces DATA_CONFIG + TIME_FEATURES + FEATURE_SCHEMA + INPUT_SPEC.
cfg = PanelConfig(
    # --- identity / target ---
    id_col="Id",
    target_col="Transactions",
    frequency="weekly",
    training_start="1999-01-01", training_end="2000-12-31",
    # Temporal validation window: the last year of calibration (2000), held out for
    # early stopping / model selection -- a TIME window over all customers.
    validation_start="2000-01-01",
    holdout_start="2001-01-01",  holdout_end="2001-12-31",
    time_cols=("year", "week"),
    clip_target_upper=6,

    # --- engineered time features (OPT-IN: omit -> none are created) ---
    time_features={"add_year_idx": True, "add_week_sin_cos": True},

    # --- feature roles (the target is target_col; don't list it here) ---
    # week_sin / week_cos are auto-added to the 'time' role from the enabled
    # flags above; year_idx is placed explicitly (a trend, not cyclical).
    known_future=("year_idx", "high.season"),
    static=("Gender", "Income"),

    # --- which columns to embed; "auto" infers cardinality from calibration ---
    #   Transactions -> clip_target_upper + 1 = 7
    #   Gender / Income -> (calibration max + 1)   (pin an int to fix the size)
    embedded_cols={"Transactions": "auto", "Gender": "auto", "Income": "auto"},
)

In [ ]:
# ---- Monte Carlo forecast settings ----------------------------------------
# Single source of truth for the holdout simulation count: every mc_forecast
# call below uses N_SIMULATIONS, so all models are compared on equal footing.
# Higher -> smoother per-customer mean, more compute. MC_SEED makes the
# forecast reproducible (same model + data + seed -> identical sampled paths).
N_SIMULATIONS = 100
MC_SEED = 42

In [ ]:
# ---- 1. Build the (N, T, F) arrays from your config ----------------------
panel     = pd.read_csv(csv_path)

data_full = dynamic_panel_dataset.prepare_dataset(panel, cfg)

# ---- 2. Shape tensors for fit_model --------------------------------------
# samples : (N, T-1, F) float32
# targets : fit_model wants (B, T) long with values in [0, max_trans)
X = data_full["samples"]
y = data_full["targets"].squeeze(-1).astype(np.int64)

# max_trans comes from the RESOLVED spec prepare_dataset returns (handles 'auto').
max_trans = data_full["embedded_cols"][data_full["target_col"]]
assert y.min() >= 0 and y.max() < max_trans, (
    f"Transactions in [{y.min()}, {y.max()}] but the target embedding caps at {max_trans-1}. "
    f"Raise the cardinality or clip the panel."
)

# ---- 3. Train/validation split is TEMPORAL (in the config) ---------------
# `validation_start` carves the calibration tail into the validation window for ALL
# customers; prepare_dataset records it as data_full["val_start_idx"]. make_loaders /
# make_data_builder build the loaders from it -- there is no customer-wise split.


## Transformer training & hyperparameter search

### Transformer with Optuna

In [ ]:
# ---- 6. data_builder for Optuna -----------------------------------------
# `make_data_builder` returns the closure run_optuna_study calls each trial: it
# slices data_full to the trial's feature subset and builds the temporal train/val
# DataLoaders (the split is data_full["val_start_idx"], shared by every trial).
data_builder = make_data_builder(data_full)


### Loss trials

In [ ]:
# ---- 5. (Optional) class weights for focal / weighted_ce -----------------
# Skip the next two lines for the paper-faithful CE run.
LOSS_TYPE     = "cross_entropy"            # 'cross_entropy' | 'weighted_ce' | 'focal' | 'emd'
# Dict form weights on the TRAINING PREFIX only (periods before validation_start),
# inferring num_classes (= max_trans) from the resolved target embedding.
class_weights = compute_class_weights(data_full)
print("class_weights:", class_weights.tolist())

## Optuna optimization cross entropy

In [ ]:
# ---- 7. Run the Transformer Optuna study --------------------------------
device = "cuda" if torch.cuda.is_available() else "cpu"
STUDY_NAME = f"transformer_{LOSS_TYPE}_3"            # one study per loss → no schema collisions

transformer_study = run_optuna_study(
    model_type="transformer",
    data_builder=data_builder,
    data_info={
        "n_epochs":       150,
        "patience":       7,
        "checkpoint_dir": "./checkpoints/transformer_optuna",
        "verbose":        False,
        # Loss configuration (all four read from data_info — unused keys are ignored).
        "loss_type":      LOSS_TYPE,
        "class_weights":  class_weights    # used by 'weighted_ce' / 'focal'; harmless for 'cross_entropy'
    },
    removable_features=["Gender", "Income", "high.season", "year_idx", ("week_sin", "week_cos")],  # Optuna can drop these if it wants; the LSTM will still get the seq/time features
    device=device,
    n_trials=10,                            # 64 archs × 9 dropout points × 3 batches — give TPE room
    study_name=STUDY_NAME,
    summary_dir="./optuna_summaries"
)

print("best trial   :", transformer_study.best_trial.number)
print("best val loss:", transformer_study.best_trial.value)
print("best params  :", transformer_study.best_trial.params)
print("checkpoint   :", transformer_study.best_trial.user_attrs["checkpoint_path"])


## Optuna rollout

In [ ]:
# ---- 7. Run the Transformer Optuna study --------------------------------
device = "cuda" if torch.cuda.is_available() else "cpu"
STUDY_NAME = f"transformer_{LOSS_TYPE}_rollout_composite"   # fresh name: composite score is a different scale than val-CE — never share a val_loss study's storage

transformer_study = run_optuna_study(
				model_type="transformer",
				selection_metric="rollout_composite",  # select on the autoregressive rollout, not teacher-forced val loss
				rollout_data=data_full,                # full data; the study carves a leak-free pseudo-holdout from the calibration tail internally
				rollout_n_simulations=30,              # LOWER than the LSTM's 100: the Transformer rollout is stateless / growing-window (O(t) per step), so each trial is much costlier
				data_builder=data_builder,
				data_info={
								"n_epochs":       100,
								"patience":       10,
								"checkpoint_dir": "./checkpoints/transformer_optuna",
								"verbose":        False,
								# Loss configuration (all read from data_info — unused keys are ignored).
								"loss_type":      LOSS_TYPE,
								"class_weights":  class_weights,   # used by 'weighted_ce' / 'focal'; harmless for 'cross_entropy'
				},
				removable_features=["Gender", "Income", "high.season", "year_idx", ("week_sin", "week_cos")],
				device=device,
				n_trials=30,                           # transformer rollout is slow; start modest, raise if you have time budget
				study_name=STUDY_NAME,
				summary_dir="./optuna_summaries",
				storage="sqlite:///optuna_summaries/transformer_rollout.db",  # separate DB from the LSTM study
)

print("best trial   :", transformer_study.best_trial.number)
print("best val loss:", transformer_study.best_trial.value)
print("best params  :", transformer_study.best_trial.params)
print("checkpoint   :", transformer_study.best_trial.user_attrs["checkpoint_path"])

In [ ]:
# ---- 8. Rebuild the Transformer with the Optuna-selected arch + load weights ----
# `build_inference_from_trial` reads the best trial's architecture + checkpoint,
# slices data_full to that trial's feature subset, rebuilds the matching inference
# model (always samples), and loads the weights (handling the Transformer's
# `_cached_mask` buffer). It returns BOTH the model and the sliced `data_best`.
inference_model, data_best = build_inference_from_trial(transformer_study, data_full, "transformer")

In [ ]:
# ---- 8b. (Optional) Final retrain on the FULL calibration window ----------
# Warm-start the selected Transformer and fine-tune a few big-batch epochs on the
# FULL calibration window (validation tail included) -- Valendin et al.'s paper step.
# Rebinds inference_model / data_best with the refit weights.
#
# Tune the retrain here:
REFIT_ON_FULL_CALIBRATION = True   # False -> keep the tuning checkpoint as-is (no retrain)
REFIT_BATCH_SIZE = 512             # the paper's "big batch"; raise/lower to taste
REFIT_N_EPOCHS   = None            # None -> a few epochs (default 5); set an int
                                   #         for a fixed number of epochs

if REFIT_ON_FULL_CALIBRATION:
    inference_model, data_best = refit_best_trial(
        transformer_study, data_full, "transformer",
        n_epochs=REFIT_N_EPOCHS,
        batch_size=REFIT_BATCH_SIZE,
        device=device,
        checkpoint_dir="./checkpoints/transformer_optuna",
    )


In [ ]:
# ---- 9. Valendin-style autoregressive Monte Carlo forecast ---------------
# The Transformer is STATELESS: unlike the LSTM (which threads a hidden state
# forward), it has to re-attend over the actual history at every step. So the
# Transformer rollout uses a GROWING context window -- `mc_forecast_transformer`
# warms up on the full calibration window, then for each holdout period re-feeds
# [calibration + already-sampled holdout] and reads the last position. This keeps
# the positional encoding consistent with training (calibration at 0..T_CAL-1,
# holdout step t at T_CAL+t). True holdout counts are never fed back -- only the
# model's own samples are (no leakage).
#
# Feed `data_best` (the winning trial's feature slice from build_inference_from_trial),
# so the forecaster's seq_cols/target_idx match the trained weights.
forecast = mc_forecast_transformer(
    inference_model,
    data_best,
    n_simulations=N_SIMULATIONS,   # average this many sampled paths -> expected count
    device=device,
    seed=MC_SEED,                  # reproducible: same model + data + seed -> identical paths
)

# Guard against a stale (not-re-run) cell silently reporting an old count.
assert forecast["simulations"].shape[0] == forecast["n_simulations"] == N_SIMULATIONS

print("simulations shape:", forecast["simulations"].shape)        # (S, N, T_HOLD)
print("prediction mean  :", forecast["prediction_mean"].shape)    # (N, T_HOLD)
print("actual (real)    :", forecast["actual"].shape)             # (N, T_HOLD)

In [ ]:
# ---- 10. Score + sanity check -------------------------------------------
metrics = mc_compute_metrics(forecast["actual"], forecast["prediction_mean"])
print(metrics)

# Aggregate predicted vs actual per week (useful for the thesis plot).
agg_pred   = forecast["prediction_mean"].sum(axis=0)              # (T_HOLD,)
agg_actual = forecast["actual"].sum(axis=0)
for i in range(min(20, len(agg_pred))):
    print(f"  week {i:>2}  pred={agg_pred[i]:6.1f}  actual={agg_actual[i]:6.1f}")

In [ ]:
# Aggregate actuals across customers per holdout week.
actuals = forecast["actual"].sum(axis=0)            # (T_HOLD,)

# (Optional) training-window aggregate to show context to the left of the holdout.
# data_full["calibration"] is (N, T_CAL, F); the target column lives at target_idx.
target_idx    = data_full["seq_cols"].index(data_full["target_col"])
train_actuals = data_full["calibration"][..., target_idx].sum(axis=0)   # (T_CAL,)

# Add a trailing 1-axis so weekly_aggregate_predictions takes the (S, N, T, 1)
# branch and draws the 95% MC ribbon.
mc_sims = forecast["simulations"][..., None]        # (S, N, T_HOLD, 1)

fig, ax = plot_weekly_aggregated(
				actuals=actuals,
    data=data_best,
    pareto_paper_benchmark=True,      
				predictions_by_model={"LSTM (MC mean + 95% CI)": mc_sims},
				train_actuals=train_actuals,                    # omit to plot only the holdout
				title="Bank panel — weekly aggregate transactions",
				show_ci=True,
				# save_path="figures/bank_lstm_weekly.png",    # uncomment to save
)
